In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Training Object Detection Model - YOLO26 Nano

In [ ]:
# download the dataset from Roboflow

from roboflow import Roboflow

api_key = os.getenv("ROBOFLOW_API_KEY")

rf = Roboflow(api_key=api_key)
project = rf.workspace("david-van-kriedt-s-workspace").project("offsider")
version = project.version(5)
dataset = version.download("yolov8", location="../data/sign-detection-v3")

In [ ]:
# Training YOLO26

from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=80,
    imgsz=640,
    batch=16,
    patience=20,
    plots=True,
    project="../runs/detect",
    name="v0.3.0",
    device="mps",
)

In [ ]:
# Save the best model weights to the models directory

import shutil
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

best_weights = Path(results.save_dir) / "weights" / "best.pt"

target_weights = models_dir / "best.pt"
shutil.copy(best_weights, target_weights)
shutil.copy(best_weights, models_dir / "best_v0.3.0.pt")

print(f"Model saved to: {target_weights.resolve()}")

In [ ]:
# Test the model on the test dataset

test_model = YOLO("../models/best.pt")

test_results = test_model.predict(source="../data/sign-detection-v3/valid/images", conf=0.65, max_det=1, save=True)

test_results[0].show()

# Evaluation

In [ ]:
import IPython


IPython.display.Image(filename=f"{results.save_dir}/confusion_matrix_normalized.png")

In [ ]:
IPython.display.Image(filename=f"{results.save_dir}/BoxF1_curve.png")